# Phase 5 — model comparison and selection

Four candidates are trained consistently on `processed/train.csv`. Validation is used for model and threshold selection; `processed/test.csv` remains untouched until final scoring. The existing root `model.pkl` (trained on train + validation) is excluded from this independent comparison.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "processed" / "train.csv").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from ml.evaluation.pipeline import load_splits, build_model_specs, classification_metrics, select_threshold
splits = load_splits(ROOT)
X_train, y_train = splits["train"]
X_validation, y_validation = splits["validation"]
X_test, y_test = splits["test"]
models = build_model_specs(X_train, y_train)
rows = []
for name, estimator in models.items():
    fitted = estimator.fit(X_train, y_train)
    validation_scores = fitted.predict_proba(X_validation)[:, 1]
    threshold = select_threshold(y_validation, validation_scores)
    test_scores = fitted.predict_proba(X_test)[:, 1]
    validation_metrics = classification_metrics(y_validation, validation_scores, threshold.threshold)
    test_metrics = classification_metrics(y_test, test_scores, threshold.threshold)
    rows.append({"model": name, "validation_threshold": threshold.threshold, "validation_roc_auc": validation_metrics["roc_auc"], "validation_log_loss": validation_metrics["log_loss"], "validation_f1": validation_metrics["f1"], "test_roc_auc": test_metrics["roc_auc"], "test_log_loss": test_metrics["log_loss"], "test_f1": test_metrics["f1"]})
comparison = pd.DataFrame(rows).sort_values("validation_roc_auc", ascending=False)
comparison


## Selection rule

The primary model is selected by highest validation ROC-AUC. Thresholds are selected by validation F1. Reported test log loss and other test metrics are final estimates only; they are not used to tune the model.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison.plot.bar(x="model", y="validation_roc_auc", ax=axes[0], legend=False, title="Validation ROC-AUC")
comparison.plot.bar(x="model", y="validation_log_loss", ax=axes[1], legend=False, title="Validation log loss (lower is better)")
for ax in axes: ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()
plt.close(fig)
